In [3]:
import pdfplumber
import pandas as pd
import re

pdf_path = "C:/Users/ibuba/Downloads/DAFTAR_1.PDF" # Pastikan nama file sesuai

data = []

# Settingan baru: HANYA percaya garis tabel asli (lines), jangan nebak dari spasi text
settings = {
    "vertical_strategy": "lines", 
    "horizontal_strategy": "lines",
    "snap_tolerance": 4,    # Toleransi buat garis yang agak miring/gak nyambung dikit
    "join_tolerance": 3,    # Gabungin huruf yang spasinya agak jauh biar gak kepotong
}

with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        # Ekstrak tabel dengan settingan baru
        tables = page.extract_tables(table_settings=settings)
        
        for table in tables:
            for row in table:
                # Bersihin row dari None values
                cleaned_row = [cell.strip() if cell else "" for cell in row]
                
                # Validasi: Baris data harus punya Nomor di kolom pertama
                # Dan panjang row minimal 4 (No, Indo, Ing, Gelar)
                if cleaned_row[0].isdigit() and len(cleaned_row) >= 4:
                    
                    # 1. Ambil data mentah
                    no = cleaned_row[0]
                    # Gabungin text yang kepisah baris (newline) jadi satu spasi
                    indo = cleaned_row[1].replace('\n', ' ').strip()
                    inggris_raw = cleaned_row[2].replace('\n', ' ').strip()
                    
                    # Cek jumlah kolom. Kalau kolom Jenjang (D3/S1) kebaca terpisah (kolom ke-4), aman.
                    # Kalau cuma 4 kolom, berarti Jenjang nyatu sama Inggris.
                    if len(cleaned_row) > 4:
                        jenjang = cleaned_row[3].replace('\n', '').strip()
                        gelar = cleaned_row[4].replace('\n', '').strip()
                    else:
                        jenjang = "" # Nanti diekstrak dari regex
                        gelar = cleaned_row[3].replace('\n', '').strip()

                    # 2. Cleaning Lanjutan: Benerin kata yang mungkin kepotong "-" (misal: "Adminis- trasi")
                    indo = indo.replace('- ', '') 
                    inggris_raw = inggris_raw.replace('- ', '')

                    # 3. Logic Darurat: Ekstrak Jenjang kalau kosong
                    jenjang_pattern = r"(D3|D4|S1|S2|S3|Sp1|Profesi)"
                    
                    if not jenjang:
                        # Coba cari di kolom Inggris
                        match = re.search(jenjang_pattern, inggris_raw)
                        if match:
                            jenjang = match.group(0)
                            inggris_raw = re.sub(jenjang_pattern, "", inggris_raw)
                        else:
                            # Kadang jenjang nyasar ke kolom Gelar kalau formatnya berantakan
                            match_gelar = re.search(jenjang_pattern, gelar)
                            if match_gelar:
                                jenjang = match_gelar.group(0)
                                # Jangan hapus jenjang dari gelar, karena kadang emang nempel
                    
                    # Rapikan spasi ganda sisa regex
                    inggris = " ".join(inggris_raw.split())
                    
                    data.append([no, indo, inggris, jenjang, gelar])

# Export ke CSV
df = pd.DataFrame(data, columns=["No", "Program Studi (Ind)", "Program Studi (Ing)", "Jenjang", "Gelar"])
df.to_csv("C:/Users/ibuba/Downloads/daftar_gelar_fixed.csv", index=False)
print(f"Selesai bos! {len(df)} data berhasil diekstrak.")

Selesai bos! 1070 data berhasil diekstrak.
